# HintFlow DPO-v2 128 EM（Colab 单卡 G4 · 96GB）

前提：已连上 Colab runtime，单卡约 **96GB**。

同卡起 OSS + orch → smoke test（必须过）→ 128 EM。
**Runtime → Run all**。

若 `n_error≈128` / EM=0 且只跑了约 1 分钟：那是 API 全挂了，不是真成绩；看 smoke / `sample_error` / `logs/*.log`。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess, time, json
from pathlib import Path

WORKDIR = Path("/content/PAgent")
REPO = "https://github.com/Ivann1242/PAgent.git"
OSS_ID = "openai/gpt-oss-20b"
ORCH_ID = "ivaning0919/pagent-hintflow-dpo-v2-merged"
OSS_PORT, ORCH_PORT = 8006, 8086
GPU = "0"
# 同卡：util 按整卡算、之和 < 1；max_len 别太大，否则 KV OOM → 全 error
OSS_UTIL, ORCH_UTIL = "0.58", "0.18"
OSS_MAX_LEN, ORCH_MAX_LEN = "16384", "8192"
MAX_NUM_SEQS = "4"
WORKERS = 4
print("WORKDIR=", WORKDIR)

In [ ]:
if not (WORKDIR / "HintFlow" / "eval_hintflow.py").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(WORKDIR)])
else:
    # 已有目录则拉最新（含 sample_error 打印）
    subprocess.call(["git", "-C", str(WORKDIR), "pull", "--ff-only"])
os.chdir(WORKDIR)
!git -C /content/PAgent rev-parse --short HEAD

In [ ]:
%pip install -q -r requirements.txt huggingface_hub "openai>=1.40,<2"
%pip install -q "vllm>=0.10"

In [ ]:
from huggingface_hub import snapshot_download

oss_dir = WORKDIR / "models" / "gpt-oss-20b"
orch_dir = WORKDIR / "checkpoints" / "hintflow_dpo_v2_merged"
oss_dir.parent.mkdir(parents=True, exist_ok=True)
orch_dir.parent.mkdir(parents=True, exist_ok=True)

print("downloading OSS…")
snapshot_download(OSS_ID, local_dir=str(oss_dir))
print("downloading orch…")
snapshot_download(ORCH_ID, local_dir=str(orch_dir))
print("ok")

In [ ]:
LOG = WORKDIR / "logs"
LOG.mkdir(exist_ok=True)

def kill_port(port: int):
    subprocess.run(
        f"pkill -f 'vllm.entrypoints.openai.api_server.*--port {port}' || true",
        shell=True,
    )

def wait_model(port: int, name: str, timeout=1200):
    import urllib.request
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{port}/v1/models", timeout=3) as r:
                body = r.read().decode()
                if name in body:
                    print(f":{port} ready ({name})")
                    return
        except Exception:
            pass
        time.sleep(5)
    raise RuntimeError(f"timeout :{port} — !tail -80 {LOG}/oss-8006.log 或 orch-8086.log")

kill_port(OSS_PORT); kill_port(ORCH_PORT); time.sleep(3)

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = GPU

oss_log = open(LOG / "oss-8006.log", "w")
orch_log = open(LOG / "orch-8086.log", "w")

oss_cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", str(oss_dir), "--port", str(OSS_PORT),
    "--tensor-parallel-size", "1",
    "--gpu-memory-utilization", OSS_UTIL,
    "--max-model-len", OSS_MAX_LEN,
    "--max-num-seqs", MAX_NUM_SEQS,
    "--served-model-name", "gpt-oss-20b",
]
orch_cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", str(orch_dir), "--port", str(ORCH_PORT),
    "--tensor-parallel-size", "1",
    "--gpu-memory-utilization", ORCH_UTIL,
    "--max-model-len", ORCH_MAX_LEN,
    "--max-num-seqs", MAX_NUM_SEQS,
    "--served-model-name", "qwen3-4b",
]

print("starting OSS…", flush=True)
oss_p = subprocess.Popen(oss_cmd, env=env, cwd=str(WORKDIR), stdout=oss_log, stderr=subprocess.STDOUT)
wait_model(OSS_PORT, "gpt-oss-20b")

print("starting orch…", flush=True)
orch_p = subprocess.Popen(orch_cmd, env=env, cwd=str(WORKDIR), stdout=orch_log, stderr=subprocess.STDOUT)
wait_model(ORCH_PORT, "qwen3-4b")
print("servers up", oss_p.pid, orch_p.pid)
!nvidia-smi

In [ ]:
# smoke：必须能真正 completion，否则别跑 128
from openai import OpenAI

def smoke(url, model, prompt="Say hi in 3 words."):
    c = OpenAI(base_url=url, api_key="EMPTY", timeout=120.0)
    r = c.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=64,
        temperature=0,
    )
    msg = r.choices[0].message
    text = (msg.content or "") or getattr(msg, "reasoning", None) or getattr(msg, "reasoning_content", None) or ""
    print(model, "->", repr(str(text)[:200]))
    return str(text)

try:
    smoke(f"http://127.0.0.1:{OSS_PORT}/v1", "gpt-oss-20b")
    smoke(f"http://127.0.0.1:{ORCH_PORT}/v1", "qwen3-4b")
    print("SMOKE OK")
except Exception as e:
    print("SMOKE FAILED:", type(e).__name__, e)
    print("--- OSS log ---")
    !tail -60 /content/PAgent/logs/oss-8006.log
    print("--- orch log ---")
    !tail -60 /content/PAgent/logs/orch-8086.log
    raise

In [ ]:
data = WORKDIR / "data" / "DAPO-Math.parquet"
assert data.exists(), f"missing {data}"
out = WORKDIR / "checkpoints" / "eval_hintflow_dpo_v2_128"

!python HintFlow/eval_hintflow.py \
  --data-file {data} \
  --limit 128 \
  --workers {WORKERS} \
  --solver-urls http://127.0.0.1:{OSS_PORT}/v1 \
  --orch-url http://127.0.0.1:{ORCH_PORT}/v1 \
  --orch-model qwen3-4b \
  --solver-model gpt-oss-20b \
  --out-dir {out}

summary = json.loads((out / "summary.json").read_text())
print(json.dumps(summary, indent=2))
if summary.get("live_baseline", {}).get("n_error", 0) or summary.get("hintflow", {}).get("n_error", 0):
    print("\n!! 有 error，先看 sample_error；真 EM 不能信。")
    !tail -40 /content/PAgent/logs/oss-8006.log

In [ ]:
# 可选：停服务
# kill_port(OSS_PORT); kill_port(ORCH_PORT)